# Lilylet patchifier visualization

This notebook loads `demo-BWV610.lyl`, converts it to NotaGen-style Lilylet patches with `starry.lilylet.data.patchifier`, and prints a readable view of the resulting patch grid.

In [1]:
from pathlib import Path
import sys

# Notebook is stored under deep-starry/tests. Add repo root to sys.path.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'tests' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from starry.lilylet.data.patchifier import (
    LilyletTokenizer,
    patchify_text,
    split_lilylet_document,
    split_measures,
)

TOKENIZER_PATH = REPO_ROOT / 'assets' / 'manual-tokenizer.json'

print('repo:', REPO_ROOT)
print('tokenizer:', TOKENIZER_PATH)

repo: /home/camus/work/deep-starry
tokenizer: /home/camus/work/deep-starry/assets/manual-tokenizer.json


In [2]:
text = r'''[title "Jesu, meine Freude"]
[subtitle "BWV 610"]
[composer "J.S. Bach"]

\staff "1" \key c \minor \time 4/4 \clef "treble" \stemUp g'4 g f ef \\
\staff "1" \stemDown ef16[ d ef8]~ ef16[ f ef d] c8[ d]~ d[ c] \\
\staff "2" \clef "bass" c16[ b c8]~ c16[ b c g] a8[ g]~ g16[ g af ef] \\
\staff "3" \clef "bass" r8 c,16[ d] ef[ d ef8]~ ef16[ a, b g] c[ b c8] | % 1

\staff "1" \stemUp d2 c\fermata \\
\staff "1" \stemDown c8[ c4 b8] c8.[ \staff "2" \stemUp g16] \staff "1" c[ b c d] \\
\staff "2" f,16[ ef f d] g[ af g f] ef[ d ef8]~ ef16[ f ef d] \\
\staff "3" r16 g,[ af f] g[ f g8] c,2 \bar "|." | % 2
'''

tokenizer = LilyletTokenizer(str(TOKENIZER_PATH))

metadata_lines, body_lines = split_lilylet_document(text)
measures = split_measures(body_lines)

print(f'input chars: {len(text)}')
print(f'metadata lines: {len(metadata_lines)}')
print(f'body lines: {len(body_lines)}')
print(f'measures: {len(measures)}')
print('--- metadata ---')
print(''.join(metadata_lines) or '(none)')
print('--- first measure ---')
print(measures[0] if measures else '(none)')

input chars: 602
metadata lines: 3
body lines: 8
measures: 2
--- metadata ---
[title "Jesu, meine Freude"]
[subtitle "BWV 610"]
[composer "J.S. Bach"]

--- first measure ---
\staff "1" \key c \minor \time 4/4 \clef "treble" \stemUp g'4 g f ef \\
\staff "1" \stemDown ef16[ d ef8]~ ef16[ f ef d] c8[ d]~ d[ c] \\
\staff "2" \clef "bass" c16[ b c8]~ c16[ b c g] a8[ g]~ g16[ g af ef] \\
\staff "3" \clef "bass" r8 c,16[ d] ef[ d ef8]~ ef16[ a, b g] c[ b c8] |



In [3]:
PATCH_SIZE = 16
PATCH_LENGTH = 2048
PATCH_STREAM = True

patches, mask, unknowns = patchify_text(
    text,
    tokenizer,
    file='demo-BWV610.lyl',
    patch_size=PATCH_SIZE,
    patch_length=PATCH_LENGTH,
    patch_stream=PATCH_STREAM,
)

print('patches shape:', tuple(patches.shape))
print('mask shape:', tuple(mask.shape))
print('real patch count:', int(mask.sum()))
print('unknown total:', sum(hit['count'] for hit in unknowns))
unknowns[:10]

patches shape: (35, 16)
mask shape: (35,)
real patch count: 35
unknown total: 0


[]

In [4]:
id_to_token = {entry['id']: entry['token'] for entry in tokenizer.vocab}
special_ids = {tokenizer.pad_id, tokenizer.bos_id, tokenizer.eos_id, tokenizer.unknown_id}

def display_token(token_id: int) -> str:
    token = id_to_token.get(token_id)
    if token is None:
        return f'<id:{token_id}>'
    if token == '\n':
        return r'\n'
    if token == '\t':
        return r'\t'
    if token == ' ':
        return '·'
    if token_id == tokenizer.pad_id:
        return '<pad>'
    return token

def show_patch(index: int):
    ids = [int(x) for x in patches[index].tolist()]
    rendered = [display_token(x) for x in ids]
    print(f'patch {index:04d} | mask={int(mask[index])}')
    print('ids:   ', ids)
    print('tokens:', ' | '.join(rendered))

for i in range(min(24, patches.shape[0])):
    show_patch(i)
    print('-' * 100)

patch 0000 | mask=1
ids:    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2]
tokens: <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <eos>
----------------------------------------------------------------------------------------------------
patch 0001 | mask=1
ids:    [91, 116, 105, 116, 108, 101, 32, 34, 74, 101, 115, 117, 44, 32, 109, 101]
tokens: [ | t | i | t | l | e | · | " | J | e | s | u | , | · | m | e
----------------------------------------------------------------------------------------------------
patch 0002 | mask=1
ids:    [105, 110, 101, 32, 70, 114, 101, 117, 100, 101, 34, 93, 10, 2, 0, 0]
tokens: i | n | e | · | F | r | e | u | d | e | " | ] | \n | <eos> | <pad> | <pad>
----------------------------------------------------------------------------------------------------
patch 0003 | mask=1
ids:    [91, 115, 117, 98, 116, 105, 116, 108, 101, 32, 34, 66, 87, 86, 32, 54]
tokens: [ | s | u | b | t | i | 

In [5]:
# Compact table view for the first N patches.
N = min(80, patches.shape[0])
for i in range(N):
    rendered = [display_token(int(x)) for x in patches[i].tolist()]
    print(f'{i:04d}  ' + ' '.join(f'{tok:>12}' for tok in rendered))

0000         <bos>        <bos>        <bos>        <bos>        <bos>        <bos>        <bos>        <bos>        <bos>        <bos>        <bos>        <bos>        <bos>        <bos>        <bos>        <eos>
0001             [            t            i            t            l            e            ·            "            J            e            s            u            ,            ·            m            e
0002             i            n            e            ·            F            r            e            u            d            e            "            ]           \n        <eos>        <pad>        <pad>
0003             [            s            u            b            t            i            t            l            e            ·            "            B            W            V            ·            6
0004             1            0            "            ]           \n        <eos>        <pad>        <pad>        <pad>        <pad>        <